# 01. Эксперименты с моделью-генератором SQL

**Цель этапа:** подключить LLM, собрать первую версию промпта и контекста (DDL-схема + базовые правила безопасности), прогнать baseline на 5-7 тестовых запросах, провести первые эксперименты, залогировать всё для последующего анализа.

**Стабилизированные промпты будут потом перенесены в `src/generator/prompts.py`.**

## Структура
1. Окружение и LLM-клиент
2. Парсер DDL-схемы → текстовое представление для промпта
3. Системный промпт v0 (без few-shot, baseline)
4. Тест-кейсы
5. Прогон по моделям
6. Анализ логов в pandas

In [12]:
# %pip install sentence-transformers

## 1. Окружение и LLM-клиент

Перед запуском убедиться, что:
- Создан файл `.env` (скопировать из `.env.example`) и заполнен `OPENROUTER_API_KEY`.
- Установлены зависимости: `pip install -r requirements.txt`.
- Для локальных моделей: запущена Ollama (`ollama serve`) и скачана модель (`ollama pull qwen2.5-coder:7b`).

In [17]:
"""
Окружение: подключаем .env, определяем пути.
LLMClient встроен прямо в ноутбук (ячейка ниже) -- чтобы ноутбук был
самодостаточным. Когда LLMClient переедет в app/services/_shared/,
эта ячейка будет заменена на импорт.
"""
import os
from pathlib import Path
from dotenv import load_dotenv

# Корень проекта -- папка над notebooks/
PROJECT_ROOT = Path.cwd().parent
load_dotenv(PROJECT_ROOT / ".env")

# Папка для логов экспериментов
LOG_DIR = PROJECT_ROOT / "notebooks" / "experiment_logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)


def get_api_key() -> str | None:
    """Достаёт ключ OpenRouter из окружения.

    Приоритет: переменная окружения OPENROUTER_API_KEY -> LLM_API_KEY (из .env
    проекта) -> None. Ключ НЕ хранится в ноутбуке -- иначе он попадёт в git.
    Если ключа нет, ячейки с реальными вызовами LLM выведут понятную ошибку.
    """
    return os.environ.get("OPENROUTER_API_KEY") or os.environ.get("LLM_API_KEY")


_key = get_api_key()
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"Логи: {LOG_DIR}")
if _key:
    print(f"API-ключ найден: {_key[:12]}... (длина {len(_key)})")
else:
    print(
        "API-ключ НЕ найден. Впиши OPENROUTER_API_KEY в файл .env в корне проекта, "
        "или задай переменную окружения. Без ключа ячейки с вызовом LLM не сработают."
    )

PROJECT_ROOT: /Users/egorvasilev/Files/Learning/MIPT/Studying/Семестр 2/Проектный практикум/DS-workshop2.0
Логи: /Users/egorvasilev/Files/Learning/MIPT/Studying/Семестр 2/Проектный практикум/DS-workshop2.0/notebooks/experiment_logs
API-ключ найден: sk-or-v1-305... (длина 73)


In [18]:
"""LLM-клиент, встроенный в ноутбук для самодостаточности.

Когда переедет в проект (app/services/_shared/llm_client.py) — заменим эту
ячейку на импорт. Сейчас держим тут, чтобы ноутбук работал без зависимости
от структуры проекта.
"""
import json
import os
import time
import uuid
from dataclasses import dataclass, field, asdict
from datetime import datetime, timezone
from typing import Any

from openai import OpenAI


PROVIDERS: dict[str, dict[str, str]] = {
    "openrouter": {
        "base_url": "https://openrouter.ai/api/v1",
        "api_key_env": "OPENROUTER_API_KEY",
    },
    "ollama": {
        "base_url": "http://localhost:11434/v1",
        "api_key_env": "OLLAMA_API_KEY",
        "api_key_fallback": "ollama",
    },
}


@dataclass
class LLMCallLog:
    call_id: str
    timestamp: str
    provider: str
    model: str
    temperature: float
    max_tokens: int | None
    system_prompt: str
    user_prompt: str
    response_text: str
    latency_seconds: float
    prompt_tokens: int | None = None
    completion_tokens: int | None = None
    total_tokens: int | None = None
    error: str | None = None
    metadata: dict[str, Any] = field(default_factory=dict)


class LLMClient:
    """Единый клиент для OpenAI-совместимых LLM-провайдеров."""

    def __init__(
        self,
        provider: str,
        model: str,
        api_key: str | None = None,
        log_file: Path | None = None,
        temperature: float = 0.2,
        max_tokens: int | None = 1024,
    ) -> None:
        if provider not in PROVIDERS:
            raise ValueError(f"Неизвестный провайдер: {provider}. Доступны: {list(PROVIDERS)}")
        cfg = PROVIDERS[provider]
        base_url = cfg["base_url"]
        # Приоритет ключа: явно переданный -> из env по имени провайдера -> fallback
        api_key = api_key or os.environ.get(cfg["api_key_env"]) or cfg.get("api_key_fallback")
        if not api_key:
            raise ValueError(
                f"Не задан API-ключ для {provider}. "
                f"Передай api_key или установи переменную окружения {cfg['api_key_env']}."
            )
        if provider == "ollama":
            base_url = os.environ.get("OLLAMA_BASE_URL", base_url)

        self.provider = provider
        self.model = model
        self.temperature = temperature
        self.max_tokens = max_tokens
        self.client = OpenAI(base_url=base_url, api_key=api_key)

        self.log_file = Path(log_file) if log_file else None
        if self.log_file:
            self.log_file.parent.mkdir(parents=True, exist_ok=True)

    def chat(
        self,
        user_prompt: str,
        system_prompt: str = "",
        temperature: float | None = None,
        max_tokens: int | None = None,
        metadata: dict[str, Any] | None = None,
    ) -> str:
        messages: list[dict[str, str]] = []
        if system_prompt:
            messages.append({"role": "system", "content": system_prompt})
        messages.append({"role": "user", "content": user_prompt})

        call_id = str(uuid.uuid4())[:8]
        started = time.monotonic()
        response_text = ""
        error: str | None = None
        usage_data: dict[str, int | None] = {
            "prompt_tokens": None, "completion_tokens": None, "total_tokens": None,
        }

        try:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=messages,
                temperature=temperature if temperature is not None else self.temperature,
                max_tokens=max_tokens if max_tokens is not None else self.max_tokens,
            )
            response_text = response.choices[0].message.content or ""
            if response.usage:
                usage_data = {
                    "prompt_tokens": response.usage.prompt_tokens,
                    "completion_tokens": response.usage.completion_tokens,
                    "total_tokens": response.usage.total_tokens,
                }
        except Exception as exc:
            error = f"{type(exc).__name__}: {exc}"
        finally:
            latency = time.monotonic() - started
            log = LLMCallLog(
                call_id=call_id,
                timestamp=datetime.now(timezone.utc).isoformat(),
                provider=self.provider,
                model=self.model,
                temperature=temperature if temperature is not None else self.temperature,
                max_tokens=max_tokens if max_tokens is not None else self.max_tokens,
                system_prompt=system_prompt,
                user_prompt=user_prompt,
                response_text=response_text,
                latency_seconds=round(latency, 3),
                error=error,
                metadata=metadata or {},
                **usage_data,
            )
            if self.log_file:
                with self.log_file.open("a", encoding="utf-8") as fh:
                    fh.write(json.dumps(asdict(log), ensure_ascii=False) + "\n")

        if error:
            raise RuntimeError(f"LLM call {call_id} failed: {error}")
        return response_text


print("LLMClient определён в ноутбуке. Провайдеры:", list(PROVIDERS))

LLMClient определён в ноутбуке. Провайдеры: ['openrouter', 'ollama']


In [19]:
# Фабрика клиентов под разные модели.
# Все логи генератора льются в один файл для удобного сравнения.
GENERATOR_LOG = LOG_DIR / "generator_calls.jsonl"

# Ключ берём из окружения (см. ячейку окружения). В сам ноутбук не вписываем.
_API_KEY = get_api_key()


def make_client(provider: str, model: str, **kwargs) -> LLMClient:
    return LLMClient(
        provider=provider,
        model=model,
        api_key=_API_KEY,
        log_file=GENERATOR_LOG,
        temperature=0.2,
        max_tokens=1024,
        **kwargs,
    )


# Список моделей для baseline-эксперимента.
# ВАЖНО: имена free-моделей на OpenRouter меняются, и эндпоинты периодически
# снимают. Если получаешь 404 (No endpoints found) -- проверь актуальный ID
# в каталоге https://openrouter.ai/models?max_price=0
# Если получаешь 429 (rate limit) -- бесплатный тир перегружен, повтори позже
# или используй платную модель.
MODELS_TO_TEST = [
    ("openrouter", "openai/gpt-oss-20b:free"),       # рабочая на момент проверки
    ("openrouter", "qwen/qwen3-coder-30b-a3b-instruct"), # Платная, стабильная, обученная под код
    # ("openrouter", "qwen/qwen3-coder:free"),        # под код, если доступна
    # ("openrouter", "deepseek/deepseek-r1:free"),    # рассуждающая
    # ("openrouter", "deepseek/deepseek-chat"),       # платная, стабильная
    # ("ollama", "qwen2.5-coder:7b"),                 # локально через Ollama
]
print("Будут тестироваться модели:")
for p, m in MODELS_TO_TEST:
    print(f"  {p}: {m}")

Будут тестироваться модели:
  openrouter: openai/gpt-oss-20b:free
  openrouter: qwen/qwen3-coder-30b-a3b-instruct


## 2. Парсер DDL-схемы (sqlglot)

**Работа парсера:**
1. Читает DDL-файл (`data/data_model.sql` — реальная схема заказчика).
2. Препроцессит — убирает psql-специфичные команды (`\connect`, `SET`, `CREATE DATABASE`), которые sqlglot не понимает.
3. Через AST извлекает таблицы, колонки, типы, NOT NULL, PK, комментарии.
4. Фильтрует служебные таблицы (хеш-имена `ms_*` от ORM, `pg_*` системные).
5. Помечает чувствительные поля по эвристике (имена + комментарии, англ. + русск.).

**Два формата вывода:**
- `schema_overview()` — компактный, одна строка на таблицу. Для шага выбора релевантных таблиц.
- `schema_detailed()` — полное описание выбранных таблиц. Для подкладывания в промпт генератора.


In [20]:
import re
from dataclasses import dataclass, field
import sqlglot
from sqlglot import exp


# ----------------------- структуры -----------------------

@dataclass
class ColumnInfo:
    name: str
    type: str
    nullable: bool = True
    comment: str = ""
    sensitive: bool = False
    is_pk: bool = False


@dataclass
class TableInfo:
    name: str
    schema: str = "public"
    columns: list = field(default_factory=list)
    comment: str = ""

    @property
    def qualified_name(self) -> str:
        return f"{self.schema}.{self.name}" if self.schema else self.name


# ----------------------- эвристики чувствительности -----------------------

# Английские паттерны — ищем в имени колонки.
SENSITIVE_NAME_PATTERNS = [
    r"password", r"passwd", r"hash", r"token", r"secret", r"api_key",
    r"card_number", r"card_token", r"cvv", r"pan\b",
    r"ssn", r"passport", r"inn\b", r"snils",
    r"email", r"phone", r"birthday", r"birth_date",
    r"full_name", r"fio\b",
    r"salary", r"balance",
    r"address",
]

# Русские и английские триггеры в комментариях.
SENSITIVE_COMMENT_PATTERNS = [
    r"пароль", r"пасс", r"токен", r"секрет",
    r"e-?mail", r"телефон", r"паспорт", r"снилс", r"инн\b",
    r"ФИО", r"фамилия", r"имя", r"отчество",
    r"дата\s+рожд", r"день\s+рожд",
    r"зарплат", r"оклад",
    r"адрес", r"прописк",
    r"PII", r"персональные\s+данные",
]

_name_re = re.compile("|".join(SENSITIVE_NAME_PATTERNS), re.IGNORECASE)
_comment_re = re.compile("|".join(SENSITIVE_COMMENT_PATTERNS), re.IGNORECASE)


def clean_comment(comment: str) -> str:
    """Убирает технический хвост от ORM из комментариев заказчика.

    Комментарии вида 'Кредитный договор, SysObjTypeEffective{id=..., ...}'
    обрезаются до человекочитаемой части ('Кредитный договор').
    """
    if not comment:
        return ""
    # Отрезаем всё начиная с 'SysObjTypeEffective' (технический артефакт)
    idx = comment.find("SysObjTypeEffective")
    if idx > 0:
        comment = comment[:idx]
    # Убираем висящие хвостовые запятые/пробелы
    return comment.rstrip(" ,;").strip()


def is_sensitive(column_name: str, comment: str) -> bool:
    if _name_re.search(column_name):
        return True
    if comment and _comment_re.search(comment):
        return True
    return False


# Служебные таблицы, которые не нужны модели в контексте.
SERVICE_TABLE_PATTERNS = [
    r"^ms_[0-9a-z]{20,}$",   # JPA materialized states с хеш-именами
    r"^_",
    r"^pg_",
]
_service_re = re.compile("|".join(SERVICE_TABLE_PATTERNS), re.IGNORECASE)


def is_service_table(name: str) -> bool:
    return bool(_service_re.search(name))


# ----------------------- препроцессинг psql-дампа -----------------------

def preprocess_psql_dump(text: str) -> str:
    """Убирает psql-специфичные команды, которые sqlglot не парсит."""
    lines_cleaned = []
    for line in text.splitlines():
        s = line.strip()
        if s.startswith("\\"):  # \connect и т.п.
            continue
        if s.startswith("CREATE DATABASE"):
            continue
        if s.startswith("SET "):
            continue
        lines_cleaned.append(line)
    return "\n".join(lines_cleaned)


# ----------------------- парсер -----------------------

def parse_ddl(ddl_text: str, dialect: str = "postgres") -> list:
    """Разбирает DDL в список TableInfo. Возвращает только пользовательские таблицы."""
    ddl_text = preprocess_psql_dump(ddl_text)
    statements = sqlglot.parse(ddl_text, dialect=dialect)

    tables: dict = {}  # ключ — qualified_name

    # 1. CREATE TABLE
    for stmt in statements:
        if not isinstance(stmt, exp.Create) or stmt.args.get("kind") != "TABLE":
            continue
        table_node = stmt.this
        if isinstance(table_node, exp.Schema):
            tbl = table_node.this
            cols_defs = table_node.expressions
        else:
            tbl = table_node
            cols_defs = []

        if not isinstance(tbl, exp.Table):
            continue
        name = tbl.name
        schema = tbl.db or "public"

        if is_service_table(name):
            continue

        cols = []
        for col_def in cols_defs:
            if not isinstance(col_def, exp.ColumnDef):
                continue
            cname = col_def.name
            ctype = col_def.args.get("kind")
            ctype_str = ctype.sql(dialect=dialect) if ctype else "UNKNOWN"
            nullable = True
            is_pk = False
            for constraint in col_def.constraints or []:
                kind = constraint.args.get("kind")
                if isinstance(kind, exp.NotNullColumnConstraint):
                    nullable = False
                elif isinstance(kind, exp.PrimaryKeyColumnConstraint):
                    is_pk = True
                    nullable = False
            cols.append(ColumnInfo(name=cname, type=ctype_str, nullable=nullable, is_pk=is_pk))
        tables[f"{schema}.{name}"] = TableInfo(name=name, schema=schema, columns=cols)

    # 2. COMMENT ON TABLE / COLUMN
    for stmt in statements:
        if not isinstance(stmt, exp.Comment):
            continue
        kind = stmt.args.get("kind")
        target = stmt.this
        text_node = stmt.args.get("expression")
        comment_text = text_node.this if text_node else ""
        if not isinstance(comment_text, str):
            comment_text = str(comment_text) if comment_text else ""

        if kind == "TABLE":
            if isinstance(target, exp.Table):
                qname = f"{target.db or 'public'}.{target.name}"
                if qname in tables:
                    tables[qname].comment = clean_comment(comment_text)
        elif kind == "COLUMN":
            if isinstance(target, exp.Column):
                cname = target.name
                tname_node = target.args.get("table")
                schema_node = target.args.get("db")
                schema = schema_node.name if schema_node else "public"
                table_name = tname_node.name if tname_node else None
                if table_name:
                    qname = f"{schema}.{table_name}"
                    if qname in tables:
                        for c in tables[qname].columns:
                            if c.name == cname:
                                c.comment = clean_comment(comment_text)
                                break

    # 3. Помечаем чувствительные колонки (после загрузки комментариев)
    for t in tables.values():
        for c in t.columns:
            c.sensitive = is_sensitive(c.name, c.comment)

    return list(tables.values())


# ----------------------- форматирование под промпт -----------------------

def schema_overview(tables: list[TableInfo]) -> str:
    """Компактный обзор: одна строка на таблицу. Для шага выбора релевантных."""
    lines = ["## Доступные таблицы (краткий обзор)\n"]
    for t in tables:
        sens_count = sum(1 for c in t.columns if c.sensitive)
        sens_marker = f" [!{sens_count} sens]" if sens_count else ""
        desc = (t.comment[:120] + "...") if len(t.comment) > 120 else t.comment
        lines.append(f"- `{t.qualified_name}` ({len(t.columns)} cols{sens_marker}): {desc or '(без описания)'}")
    return "\n".join(lines)


def schema_detailed(tables: list[TableInfo]) -> str:
    """Полное описание выбранных таблиц со всеми колонками."""
    lines = []
    for t in tables:
        lines.append(f"### Таблица `{t.qualified_name}` ###")
        if t.comment:
            lines.append(f"({t.comment})")
        lines.append("")
        lines.append("| Колонка | Тип | NULL | PK | Sensitive | Комментарий |")
        lines.append("|---|---|---|---|---|---|")
        for c in t.columns:
            null = "" if c.nullable else "NO"
            pk = "PK" if c.is_pk else ""
            sens = "[!]" if c.sensitive else ""
            comment = c.comment.replace("|", "\\|").replace("\n", " ")
            lines.append(f"| {c.name} | {c.type} | {null} | {pk} | {sens} | {comment} |")
        lines.append("")
    return "\n".join(lines)


# === Загрузка реальной схемы заказчика ===
DDL_PATH = PROJECT_ROOT / "data_model.sql"
ddl_text = DDL_PATH.read_text(encoding="utf-8")
all_tables = parse_ddl(ddl_text)

# Индекс по qualified_name — пригодится для подвыборки
tables_by_qname = {t.qualified_name: t for t in all_tables}

print(f"Распарсено таблиц: {len(all_tables)}")
print(f"Из них с чувствительными полями: {sum(1 for t in all_tables if any(c.sensitive for c in t.columns))}")

print("\n--- Топ-10 таблиц по числу колонок ---")
for t in sorted(all_tables, key=lambda x: len(x.columns), reverse=True)[:10]:
    sens = sum(1 for c in t.columns if c.sensitive)
    sens_mark = f", [!]{sens}" if sens else ""
    print(f"  {t.qualified_name:45s} {len(t.columns):3d} кол.{sens_mark}: {t.comment[:60]}")

# Сравнение размеров
overview_text = schema_overview(all_tables)
full_text = schema_detailed(all_tables)
print(f"\nКомпактный обзор: {len(overview_text)} симв. (~{len(overview_text)//4} ток.)")
print(f"Полная схема:     {len(full_text)} симв. (~{len(full_text)//4} ток.)")


Распарсено таблиц: 47
Из них с чувствительными полями: 8

--- Топ-10 таблиц по числу колонок ---
  public.scp_project_ans                        120 кол.: СКП. Проект решения
  public.scp_application                        104 кол.: СКП. Заявка
  public.corp_tech_application                   84 кол.: КТ. Заявка
  public.dict_product                            82 кол.: Справочник: Продукт
  public.credit_contract                         76 кол., [!]4: Кредитный договор
  public.scp_collateral_app                      69 кол., [!]3: СКП. Залоговая заявка
  public.product_pricing                         66 кол.: СКП. Ценообразование по продукту
  public.scp_amd_product                         62 кол.: СКП. РУМ. Продукт РУМ
  public.mler_application                        57 кол.: МЮЭР. Заявка
  public.participant_app                         53 кол.: Участники сделки

Компактный обзор: 3372 симв. (~843 ток.)
Полная схема:     131675 симв. (~32918 ток.)


### 2b. Подвыборка релевантных таблиц

**Стратегия для baseline:** простая эвристика по ключевым словам. На вход — текст пользовательского запроса, на выход — топ-K таблиц, у которых:
- имя содержит ключевое слово из запроса (после нормализации),
- или комментарий таблицы содержит ключевое слово.

**ToDo:** Заменить на эмбеддинги или на выбирающий LLM-агент.


In [21]:
# """Селектор релевантных таблиц с TF-IDF-взвешиванием.

# Улучшение против фиксированных весов: слова, встречающиеся во многих таблицах
# (дата, статус, объект), весят меньше; редкие и специфичные (кредитный,
# залоговый, сотрудник) -- больше. Это поднимает действительно релевантную
# таблицу над "шумом".
# """
# import math
# import re
# from collections import Counter

# RU_ENDINGS = [
#     "ами", "ями", "ого", "его", "ому", "ему", "ой", "ей", "ую", "юю",
#     "ах", "ях", "ам", "ям", "ов", "ев", "ы", "и", "а", "я", "о", "е", "у", "ю",
# ]
# EN_ENDINGS = ["ing", "ed", "es", "s"]

# STOPWORDS = {
#     "и", "в", "не", "на", "с", "по", "для", "за", "от", "до", "у", "о", "из",
#     "как", "что", "это", "все", "его", "её", "их", "тот", "та", "то",
#     "найди", "получи", "покажи", "выведи", "верни", "сделай", "создай",
#     "обнови", "удали", "вставь", "посчитай", "сколько", "каждый", "каждой",
#     "запрос", "данные", "таблица", "поле", "значение", "список",
#     "последний", "первый", "новый", "старый",
#     "the", "a", "an", "of", "in", "on", "at", "to", "for", "with", "from",
#     "show", "find", "get", "list", "select", "update", "delete", "insert",
#     "all", "any", "by", "and", "or", "not", "is", "are", "count",
# }


# def _normalize_word(w: str) -> str:
#     w = w.lower()
#     if len(w) <= 3:
#         return w
#     for end in RU_ENDINGS + EN_ENDINGS:
#         if w.endswith(end) and len(w) - len(end) >= 3:
#             return w[: -len(end)]
#     return w


# def extract_keywords(text: str) -> list[str]:
#     words = re.findall(r"[a-zA-Zа-яА-ЯёЁ_]{3,}", text)
#     return [_normalize_word(w) for w in words if w.lower() not in STOPWORDS]


# class TableSelector:
#     """Выбирает релевантные таблицы под запрос (с idf-взвешиванием)."""

#     WEIGHT_TABLE_NAME = 3.0
#     WEIGHT_TABLE_COMMENT = 2.0
#     WEIGHT_COLUMN_NAME = 1.0
#     WEIGHT_COLUMN_COMMENT = 1.0

#     def __init__(self, tables: list[TableInfo]) -> None:
#         self._tables = tables
#         self._idf: dict[str, float] = {}
#         self._table_tokens: dict[str, dict[str, set[str]]] = {}
#         self._fit()

#     def _fit(self) -> None:
#         n_tables = len(self._tables) or 1
#         df: Counter = Counter()
#         for t in self._tables:
#             name_tokens = {_normalize_word(t.name)}
#             comment_tokens = set(extract_keywords(t.comment))
#             col_name_tokens = {_normalize_word(c.name) for c in t.columns}
#             col_comment_tokens: set[str] = set()
#             for c in t.columns:
#                 col_comment_tokens.update(extract_keywords(c.comment))
#             self._table_tokens[t.qualified_name] = {
#                 "table_name": name_tokens,
#                 "table_comment": comment_tokens,
#                 "column_name": col_name_tokens,
#                 "column_comment": col_comment_tokens,
#             }
#             all_words = name_tokens | comment_tokens | col_name_tokens | col_comment_tokens
#             for w in all_words:
#                 df[w] += 1
#         for word, freq in df.items():
#             self._idf[word] = math.log(n_tables / (1 + freq)) + 1.0

#     def _word_weight(self, word: str) -> float:
#         return self._idf.get(word, 1.0)

#     def _score(self, query: str) -> dict[str, float]:
#         keywords = extract_keywords(query)
#         scores: dict[str, float] = {}
#         if not keywords:
#             return scores
#         for t in self._tables:
#             zones = self._table_tokens[t.qualified_name]
#             score = 0.0
#             for kw in keywords:
#                 w = self._word_weight(kw)
#                 if kw in zones["table_name"]:
#                     score += self.WEIGHT_TABLE_NAME * w
#                 if kw in zones["table_comment"]:
#                     score += self.WEIGHT_TABLE_COMMENT * w
#                 if kw in zones["column_name"]:
#                     score += self.WEIGHT_COLUMN_NAME * w
#                 if kw in zones["column_comment"]:
#                     score += self.WEIGHT_COLUMN_COMMENT * w
#             if score > 0:
#                 scores[t.qualified_name] = score
#         return scores

#     def select(self, query: str, top_k: int = 5) -> list[TableInfo]:
#         scores = self._score(query)
#         if not scores:
#             return self._tables[:top_k]
#         by_qname = {t.qualified_name: t for t in self._tables}
#         ranked = sorted(scores.items(), key=lambda kv: kv[1], reverse=True)
#         return [by_qname[q] for q, _ in ranked[:top_k]]

#     def select_with_scores(self, query: str, top_k: int = 5) -> list[tuple[TableInfo, float]]:
#         scores = self._score(query)
#         by_qname = {t.qualified_name: t for t in self._tables}
#         ranked = sorted(scores.items(), key=lambda kv: kv[1], reverse=True)
#         return [(by_qname[q], round(s, 2)) for q, s in ranked[:top_k]]


# # Строим селектор один раз на загруженной схеме
# selector = TableSelector(all_tables)

# # Демо: подбор таблиц под несколько запросов
# DEMO_QUERIES = [
#     "Найди все активные кредитные договоры за последний месяц",
#     "Покажи список сотрудников с их email и телефонами",
#     "Сколько кредитных договоров у каждого подразделения",
# ]
# for q in DEMO_QUERIES:
#     print(f"\nЗапрос: {q}")
#     for t, s in selector.select_with_scores(q, top_k=5):
#         print(f"  {s:6.2f}  {t.qualified_name}")

In [13]:
"""Модель эмбеддингов для семантического слоя селектора.

Multilingual-модель: запросы на русском, часть комментариев схемы на английском.
Загружается один раз (скачается при первом запуске).
"""
from sentence_transformers import SentenceTransformer
import numpy as np

# paraphrase-multilingual-MiniLM-L12-v2 — лёгкая (118M), multilingual (50+ языков),
# хороша для коротких текстов. Качается один раз, далее берется из кеша.
_EMB_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
emb_model = SentenceTransformer(_EMB_MODEL_NAME)
print(f"Модель эмбеддингов загружена: {_EMB_MODEL_NAME}")


def cosine_sim(a: np.ndarray, b: np.ndarray) -> float:
    """Косинусное сходство двух векторов."""
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))


# Быстрая проверка: близость слов с разными корнями
_demo = emb_model.encode(["сотрудник", "работник", "продукт", "employee"])
print(f"сотрудник <-> работник:  {cosine_sim(_demo[0], _demo[1]):.3f}")
print(f"сотрудник <-> employee:  {cosine_sim(_demo[0], _demo[3]):.3f}")
print(f"сотрудник <-> продукт:   {cosine_sim(_demo[0], _demo[2]):.3f}")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Модель эмбеддингов загружена: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
сотрудник <-> работник:  0.861
сотрудник <-> employee:  0.914
сотрудник <-> продукт:   0.444


In [14]:
"""Гибридный селектор таблиц: лексика (TF-IDF) + семантика (эмбеддинги).

Лексический слой: точное совпадение слов с весами и idf.
Семантический слой: косинусная близость эмбеддинга запроса к эмбеддингу
описания таблицы. Ловит смысл при разных формулировках.

Итоговый скор = alpha * lexical_norm + beta * semantic_norm.
Оба слоя нормируются в [0,1], чтобы веса были сопоставимы.
"""
import math
import re
from collections import Counter
import numpy as np

RU_ENDINGS = [
    "ами", "ями", "ого", "его", "ому", "ему", "ой", "ей", "ую", "юю",
    "ах", "ях", "ам", "ям", "ов", "ев", "ы", "и", "а", "я", "о", "е", "у", "ю",
]
EN_ENDINGS = ["ing", "ed", "es", "s"]

STOPWORDS = {
    "и", "в", "не", "на", "с", "по", "для", "за", "от", "до", "у", "о", "из",
    "как", "что", "это", "все", "его", "её", "их", "тот", "та", "то",
    "найди", "получи", "покажи", "выведи", "верни", "сделай", "создай",
    "обнови", "удали", "вставь", "посчитай", "сколько", "каждый", "каждой",
    "запрос", "данные", "таблица", "поле", "значение", "список",
    "последний", "первый", "новый", "старый",
    "the", "a", "an", "of", "in", "on", "at", "to", "for", "with", "from",
    "show", "find", "get", "list", "select", "update", "delete", "insert",
    "all", "any", "by", "and", "or", "not", "is", "are", "count",
}


def _normalize_word(w: str) -> str:
    w = w.lower()
    if len(w) <= 3:
        return w
    for end in RU_ENDINGS + EN_ENDINGS:
        if w.endswith(end) and len(w) - len(end) >= 3:
            return w[: -len(end)]
    return w


def extract_keywords(text: str) -> list[str]:
    words = re.findall(r"[a-zA-Zа-яА-ЯёЁ_]{3,}", text)
    return [_normalize_word(w) for w in words if w.lower() not in STOPWORDS]


def _table_semantic_text(t: TableInfo) -> str:
    """Текст таблицы для эмбеддинга: имя + комментарий + ключевые колонки.

    Не берутся все колонки (их бывает 100+) — только имя, описание и
    комментарии колонок (достаточно для смысловой близости).
    """
    parts = [t.name, t.comment]
    col_comments = [c.comment for c in t.columns if c.comment][:20]
    parts.extend(col_comments)
    return " ".join(p for p in parts if p)


class HybridTableSelector:
    """Лексический + семантический выбор релевантных таблиц."""

    WEIGHT_TABLE_NAME = 3.0
    WEIGHT_TABLE_COMMENT = 2.0
    WEIGHT_COLUMN_NAME = 1.0
    WEIGHT_COLUMN_COMMENT = 1.0

    def __init__(self, tables, emb_model, alpha: float = 0.5, beta: float = 0.5):
        self._tables = tables
        self._emb = emb_model
        self.alpha = alpha   # вес лексики
        self.beta = beta     # вес семантики
        self._idf = {}
        self._table_tokens = {}
        self._table_vecs = None
        self._fit()

    def _fit(self):
        # лексический слой
        n_tables = len(self._tables) or 1
        df = Counter()
        for t in self._tables:
            name_tokens = {_normalize_word(t.name)}
            comment_tokens = set(extract_keywords(t.comment))
            col_name_tokens = {_normalize_word(c.name) for c in t.columns}
            col_comment_tokens = set()
            for c in t.columns:
                col_comment_tokens.update(extract_keywords(c.comment))
            self._table_tokens[t.qualified_name] = {
                "table_name": name_tokens,
                "table_comment": comment_tokens,
                "column_name": col_name_tokens,
                "column_comment": col_comment_tokens,
            }
            for w in name_tokens | comment_tokens | col_name_tokens | col_comment_tokens:
                df[w] += 1
        for word, freq in df.items():
            self._idf[word] = math.log(n_tables / (1 + freq)) + 1.0

        # семантический слой: эмбеддинги таблиц (запускается один раз при старте)
        texts = [_table_semantic_text(t) for t in self._tables]
        self._table_vecs = self._emb.encode(texts, normalize_embeddings=True)

    def _lexical_scores(self, query: str) -> dict[str, float]:
        keywords = extract_keywords(query)
        scores = {}
        if not keywords:
            return scores
        for t in self._tables:
            zones = self._table_tokens[t.qualified_name]
            s = 0.0
            for kw in keywords:
                w = self._idf.get(kw, 1.0)
                if kw in zones["table_name"]:
                    s += self.WEIGHT_TABLE_NAME * w
                if kw in zones["table_comment"]:
                    s += self.WEIGHT_TABLE_COMMENT * w
                if kw in zones["column_name"]:
                    s += self.WEIGHT_COLUMN_NAME * w
                if kw in zones["column_comment"]:
                    s += self.WEIGHT_COLUMN_COMMENT * w
            if s > 0:
                scores[t.qualified_name] = s
        return scores

    def _semantic_scores(self, query: str) -> dict[str, float]:
        qvec = self._emb.encode([query], normalize_embeddings=True)[0]
        # косинус = скалярное произведение (векторы уже нормированы)
        sims = self._table_vecs @ qvec
        return {t.qualified_name: float(sim) for t, sim in zip(self._tables, sims)}

    @staticmethod
    def _normalize(scores: dict[str, float]) -> dict[str, float]:
        if not scores:
            return {}
        vals = list(scores.values())
        lo, hi = min(vals), max(vals)
        if hi - lo < 1e-9:
            return {k: 1.0 for k in scores}
        return {k: (v - lo) / (hi - lo) for k, v in scores.items()}

    def select_with_scores(self, query: str, top_k: int = 5):
        lex = self._normalize(self._lexical_scores(query))
        sem = self._normalize(self._semantic_scores(query))
        all_names = set(lex) | set(sem)
        combined = {
            name: self.alpha * lex.get(name, 0.0) + self.beta * sem.get(name, 0.0)
            for name in all_names
        }
        by_qname = {t.qualified_name: t for t in self._tables}
        ranked = sorted(combined.items(), key=lambda kv: kv[1], reverse=True)
        return [(by_qname[q], round(s, 3)) for q, s in ranked[:top_k]]

    def select(self, query: str, top_k: int = 5):
        return [t for t, _ in self.select_with_scores(query, top_k)]


# Гибридный селектор (эмбеддинги таблиц считаются здесь, порядка 1-2 сек)
selector = HybridTableSelector(all_tables, emb_model, alpha=0.5, beta=0.5)

# Демо на проблемных запросах
DEMO_QUERIES = [
    "Найди все активные кредитные договоры за последний месяц",
    "Покажи список сотрудников с их email и телефонами",
    "Удали из системы заблокированных работников, неактивных более года",
    "Сколько кредитных договоров у каждого подразделения",
]
for q in DEMO_QUERIES:
    print(f"\nЗапрос: {q}")
    for t, s in selector.select_with_scores(q, top_k=5):
        print(f"  {s:.3f}  {t.qualified_name}")


Запрос: Найди все активные кредитные договоры за последний месяц
  1.000  public.credit_contract
  0.616  public.dict_product
  0.409  public.count_turnover
  0.389  public.dict_div_presence
  0.372  public.cb_interest_rate

Запрос: Покажи список сотрудников с их email и телефонами
  1.000  public.sys_employee
  0.580  public.sys_company
  0.368  public.offices_psb
  0.360  public.afhd_ac_trans_link
  0.344  public.acc_number

Запрос: Удали из системы заблокированных работников, неактивных более года
  0.759  public.scp_project_ans
  0.500  public.sys_employee
  0.496  public.offices_psb
  0.468  public.scp_sec_check_res
  0.458  public.sys_obj_type

Запрос: Сколько кредитных договоров у каждого подразделения
  1.000  public.credit_contract
  0.485  public.dict_product
  0.417  public.afhd_ac_trans_link
  0.405  public.scp_dict_product_na
  0.331  public.scp_application


In [27]:
# # Подбор alpha/beta без пересоздания (эмбеддинги уже посчитаны)
# for a, b in [(0.7, 0.3), (0.6, 0.4), (0.5, 0.5)]:
#     selector.alpha, selector.beta = a, b
#     print(f"\n===== alpha={a} (лексика), beta={b} (семантика) =====")
#     for q in ["Удали из системы заблокированных работников, неактивных более года",
#               "Покажи список сотрудников с их email и телефонами"]:
#         print(f"\nЗапрос: {q}")
#         for t, s in selector.select_with_scores(q, top_k=5):
#             print(f"  {s:.3f}  {t.qualified_name}")

# # вернём 0.5/0.5 по умолчанию
# selector.alpha, selector.beta = 0.5, 0.5

In [22]:
"""Хелперы для интерактивного просмотра схемы в ноутбуке.

Удобно изучать таблицы по имени, не обращаясь к all_tables по индексу.
"""

# Индекс таблиц по имени (короткому и полному) -- для быстрого доступа
_tables_by_name: dict[str, TableInfo] = {}
for _t in all_tables:
    _tables_by_name[_t.name] = _t
    _tables_by_name[_t.qualified_name] = _t


def show_table(name: str) -> None:
    """Печатает детальное описание таблицы по имени.
    Принимает короткое (sys_employee) и полное (public.sys_employee) имя.
    """
    t = _tables_by_name.get(name)
    if t is None:
        print(f"Таблица '{name}' не найдена. Доступные: см. print_overview()")
        return
    print(schema_detailed([t]))


def print_overview() -> None:
    """Печатает компактный обзор всех таблиц (имя, число колонок, чувствительные)."""
    print(schema_overview(all_tables))


def find_tables(substring: str) -> None:
    """Ищет таблицы, в имени или комментарии которых есть подстрока."""
    sub = substring.lower()
    found = [
        t for t in all_tables
        if sub in t.name.lower() or sub in t.comment.lower()
    ]
    if not found:
        print(f"Таблиц с '{substring}' не найдено.")
        return
    for t in found:
        print(f"  {t.qualified_name} ({len(t.columns)} cols): {t.comment[:60]}")


# Примеры использования (раскомментируй нужное):
# print_overview()                    # обзор всех таблиц
# show_table("sys_employee")          # детально одна таблица
# find_tables("кредит")               # поиск по подстроке
print("Хелперы готовы: show_table(name), print_overview(), find_tables(substring)")

Хелперы готовы: show_table(name), print_overview(), find_tables(substring)


## 3. Системный промпт v0 (baseline)

**Принципы baseline-промпта:**
- Жёстко задается роль и формат вывода (только SQL, без преамбул).
- Перечисляются правила безопасности списком (плейсхолдеры под параметризацию, запрет SELECT *, обязательный WHERE для UPDATE/DELETE и т.д.).
- Передается схема БД с пометками чувствительных полей.
- НЕ передаются few-shot примеры -> следующая итерация.

**ToDo:**
- Добавить few-shot примеры;
- Уточнить формулировки;
- Протестировать structured output.
- Переписать промпт на Eng.

In [28]:
SYSTEM_PROMPT_V0 = """Ты — эксперт по PostgreSQL. Твоя задача — генерировать безопасные и эффективные SQL-запросы по описанию задачи на естественном языке.

## Целевая СУБД
PostgreSQL (предполагается совместимость с версиями 12+, расширения не используем).

## Правила безопасности (СТРОГО соблюдать)

1. **Параметризация.** Все значения, приходящие от пользователя, — параметры (`$1`, `$2`, ...), НЕ конкатенация в строку запроса.
2. **WHERE обязателен для UPDATE/DELETE.** Без WHERE такие запросы изменяют/удаляют всю таблицу — это критическая ошибка.
3. **Никаких `SELECT *`.** Всегда явно перечисляй нужные колонки.
4. **Чувствительные поля.** В таблицах ниже отмечены колонки знаком <!> — это PII или секреты. НЕ включай их в выборки без явного требования. Хэши паролей, токены, ФИО, телефоны, email — НИКОГДА без явной необходимости.
5. **LIMIT и пагинация.** Запросы, потенциально возвращающие много строк (списки, поиски), должны содержать LIMIT.
6. **Никакого динамического SQL** (`EXECUTE`, `format()` с пользовательским вводом).

## Правила производительности

- Используй колонки с PK (<PK> в схеме) для джойнов.
- Избегай неявных приведений типов в условиях WHERE.
- JOIN'ы — только необходимые, не тяни лишние таблицы.
- ORDER BY без LIMIT на больших таблицах — плохая идея.

## Формат ответа

Верни ТОЛЬКО SQL-запрос. Без преамбулы, без объяснений, без markdown-блоков ```sql. Чистый SQL.

## Схема базы данных (релевантные таблицы)

{schema}
"""

# Few-shot примеры: эталоны "запрос -> безопасный SQL".
# Демонстрируют модели правила в действии: параметризация, явные колонки,
# WHERE для UPDATE/DELETE, LIMIT, осторожность с чувствительными полями.
FEW_SHOT_EXAMPLES = [
    {
        "task": "Найди пользователя по email (email приходит параметром от интерфейса).",
        "sql": "SELECT id, name, email\nFROM public.sys_employee\nWHERE email = $1\nLIMIT 1;",
    },
    {
        "task": "Обнови статус заявки на заданное значение по её ID (оба приходят от клиента).",
        "sql": "UPDATE public.corp_tech_application\nSET status = $1\nWHERE id = $2;",
    },
    {
        "task": "Покажи последние 20 заявок, новые сверху.",
        "sql": "SELECT id, name, create_date, status\nFROM public.scp_application\nORDER BY create_date DESC\nLIMIT 20;",
    },
    {
        "task": "Сколько договоров у каждого подразделения, топ-10.",
        "sql": "SELECT org_id, COUNT(*) AS cnt\nFROM public.credit_contract\nGROUP BY org_id\nORDER BY cnt DESC\nLIMIT 10;",
    },
]


def _format_few_shot(examples: list[dict]) -> str:
    """Собирает блок few-shot для вставки в промпт."""
    blocks = []
    for ex in examples:
        blocks.append(f"Запрос: {ex['task']}\nSQL:\n{ex['sql']}")
    return "\n\n".join(blocks)


def build_system_prompt(schema_text: str, with_few_shot: bool = True) -> str:
    prompt = SYSTEM_PROMPT_V0.format(schema=schema_text)
    if with_few_shot:
        prompt += (
            "\n\n## Примеры правильных запросов\n\n"
            + _format_few_shot(FEW_SHOT_EXAMPLES)
        )
    return prompt


# Демо: системный промпт для одного из тестовых запросов
demo_query = "Найди все активные кредитные договоры за последний месяц"
demo_tables = selector.select(demo_query, top_k=5)
demo_schema_md = schema_detailed(demo_tables)
demo_prompt = build_system_prompt(demo_schema_md)

print(f"Демо-запрос: {demo_query!r}")
print(f"Выбрано таблиц: {[t.qualified_name for t in demo_tables]}")
print(f"Длина системного промпта: {len(demo_prompt)} симв. (~{len(demo_prompt)//4} ток.)")
print(f"\n--- Первые 1200 символов промпта ---\n{demo_prompt[:1200]}")


Демо-запрос: 'Найди все активные кредитные договоры за последний месяц'
Выбрано таблиц: ['public.credit_contract', 'public.dict_product', 'public.count_turnover', 'public.dict_div_presence', 'public.cb_interest_rate']
Длина системного промпта: 17971 симв. (~4492 ток.)

--- Первые 1200 символов промпта ---
Ты — эксперт по PostgreSQL. Твоя задача — генерировать безопасные и эффективные SQL-запросы по описанию задачи на естественном языке.

## Целевая СУБД
PostgreSQL (предполагается совместимость с версиями 12+, расширения не используем).

## Правила безопасности (СТРОГО соблюдать)

1. **Параметризация.** Все значения, приходящие от пользователя, — параметры (`$1`, `$2`, ...), НЕ конкатенация в строку запроса.
2. **WHERE обязателен для UPDATE/DELETE.** Без WHERE такие запросы изменяют/удаляют всю таблицу — это критическая ошибка.
3. **Никаких `SELECT *`.** Всегда явно перечисляй нужные колонки.
4. **Чувствительные поля.** В таблицах ниже отмечены колонки знаком <!> — это PII или секреты. 

In [30]:
demo = build_system_prompt("(схема тут)")
print("Примеры правильных" in demo, "— few-shot в промпте")
print(demo)  # хвост промпта с примерами

True — few-shot в промпте
Ты — эксперт по PostgreSQL. Твоя задача — генерировать безопасные и эффективные SQL-запросы по описанию задачи на естественном языке.

## Целевая СУБД
PostgreSQL (предполагается совместимость с версиями 12+, расширения не используем).

## Правила безопасности (СТРОГО соблюдать)

1. **Параметризация.** Все значения, приходящие от пользователя, — параметры (`$1`, `$2`, ...), НЕ конкатенация в строку запроса.
2. **WHERE обязателен для UPDATE/DELETE.** Без WHERE такие запросы изменяют/удаляют всю таблицу — это критическая ошибка.
3. **Никаких `SELECT *`.** Всегда явно перечисляй нужные колонки.
4. **Чувствительные поля.** В таблицах ниже отмечены колонки знаком <!> — это PII или секреты. НЕ включай их в выборки без явного требования. Хэши паролей, токены, ФИО, телефоны, email — НИКОГДА без явной необходимости.
5. **LIMIT и пагинация.** Запросы, потенциально возвращающие много строк (списки, поиски), должны содержать LIMIT.
6. **Никакого динамического SQL** (`EXECU

## 4. Тест-кейсы

**Категории тест-кейсов:**
- **easy**: простые SELECT'ы (проверяем, что модель просто работает).
- **with_user_input**: задачи с пользовательским вводом -> должна быть параметризация.
- **dml**: UPDATE/DELETE -> должны быть с WHERE.
- **sensitive**: запросы, где может «соблазнить» вытащить чувствительные поля.
- **pagination**: списки/поиски -> должен быть LIMIT.

В `expected_traits` **не assert'ы**, а ожидания, по которым удобно проверять.

In [24]:
# Универсальные тест-кейсы. Завязаны не на конкретные имена таблиц из схемы,
# а на типичные сценарии работы с БД. Подвыборка таблиц делается автоматически.

TEST_CASES = [
    {
        "id": "easy_01",
        "category": "easy",
        "task": "Получи список всех продуктов с их именем и идентификатором.",
        "expected_traits": ["SELECT name", "SELECT id", "no SELECT *"],
    },
    {
        "id": "input_01",
        "category": "with_user_input",
        "task": "Найди сотрудника по email. Email передаётся параметром от пользователя интерфейса.",
        "expected_traits": ["параметризация $1", "нет конкатенации"],
    },
    {
        "id": "dml_01",
        "category": "dml",
        "task": "Обнови статус заявки на 'approved' для заявки с ID, который придёт от клиента.",
        "expected_traits": ["UPDATE", "WHERE id = $1", "status"],
    },
    {
        "id": "sensitive_01",
        "category": "sensitive",
        "task": "Покажи всю информацию о сотруднике по его ID.",
        "expected_traits": ["WHERE id = $1", "не выдавать password/токены", "осторожно с PII"],
    },
    {
        "id": "pagination_01",
        "category": "pagination",
        "task": "Покажи последние заявки на кредит, отсортированные по дате создания (новые сверху).",
        "expected_traits": ["ORDER BY ... DESC", "LIMIT"],
    },
    {
        "id": "join_01",
        "category": "easy",
        "task": "Сколько кредитных договоров у каждого подразделения. Покажи топ-10 подразделений.",
        "expected_traits": ["JOIN или подзапрос", "GROUP BY", "ORDER BY ... DESC", "LIMIT 10"],
    },
    {
        "id": "dml_02",
        "category": "dml",
        "task": "Удали из системы заблокированных сотрудников, которые не активны более года.",
        "expected_traits": ["DELETE", "WHERE есть", "is_locked / is_active"],
    },
]

print(f"Загружено тест-кейсов: {len(TEST_CASES)}")
for tc in TEST_CASES:
    print(f"  [{tc['category']:18s}] {tc['id']}: {tc['task'][:70]}")


Загружено тест-кейсов: 7
  [easy              ] easy_01: Получи список всех продуктов с их именем и идентификатором.
  [with_user_input   ] input_01: Найди сотрудника по email. Email передаётся параметром от пользователя
  [dml               ] dml_01: Обнови статус заявки на 'approved' для заявки с ID, который придёт от 
  [sensitive         ] sensitive_01: Покажи всю информацию о сотруднике по его ID.
  [pagination        ] pagination_01: Покажи последние заявки на кредит, отсортированные по дате создания (н
  [easy              ] join_01: Сколько кредитных договоров у каждого подразделения. Покажи топ-10 под
  [dml               ] dml_02: Удали из системы заблокированных сотрудников, которые не активны более


## 5. Прогон baseline по моделям

Прогонка тест-кейсов на каждой модели. Логи сохраняются в `experiment_logs/generator_calls.jsonl`.

In [25]:
from datetime import datetime

EXPERIMENT_TAG = f"baseline_v0_{datetime.now().strftime('%Y%m%d_%H%M')}"
print(f"Экспериментальный тэг: {EXPERIMENT_TAG}\n")

results = []

for provider, model in MODELS_TO_TEST:
    print(f"\n{'='*70}\nМодель: {provider} :: {model}\n{'='*70}")
    try:
        client = make_client(provider, model)
    except Exception as e:
        print(f"<X> Не удалось инициализировать клиент: {e}")
        continue

    for tc in TEST_CASES:
        # Подвыборка релевантных таблиц для этого конкретного запроса
        selected = selector.select(tc["task"], top_k=5)
        schema_md_for_query = schema_detailed(selected)
        system_prompt = build_system_prompt(schema_md_for_query)

        try:
            sql = client.chat(
                user_prompt=tc["task"],
                system_prompt=system_prompt,
                metadata={
                    "experiment": EXPERIMENT_TAG,
                    "test_case_id": tc["id"],
                    "category": tc["category"],
                    "prompt_version": "v0",
                    "selected_tables": [t.qualified_name for t in selected],
                    "system_prompt_chars": len(system_prompt),
                },
            )
            status = "<OK>"
        except Exception as e:
            sql = f"ERROR: {e}"
            status = "<X>"

        results.append({
            "model": model,
            "test_id": tc["id"],
            "category": tc["category"],
            "task": tc["task"],
            "selected_tables": [t.qualified_name for t in selected],
            "sql": sql,
        })
        print(f"\n{status} [{tc['id']}] {tc['task']}")
        print(f"   Выбраны: {[t.name for t in selected]}")
        print(f"--- SQL ---\n{sql[:500]}")
        if len(sql) > 500:
            print("...")

print(f"\n\nГотово. Всего вызовов: {len(results)}. Логи: {GENERATOR_LOG}")


Экспериментальный тэг: baseline_v0_20260525_2104


Модель: openrouter :: openai/gpt-oss-20b:free

<OK> [easy_01] Получи список всех продуктов с их именем и идентификатором.
   Выбраны: ['product_pricing', 'dict_product', 'scp_amd_product', 'scp_dict_product_na', 'prod_change_params']
--- SQL ---
SELECT id, name FROM public.dict_product LIMIT 100;

<OK> [input_01] Найди сотрудника по email. Email передаётся параметром от пользователя интерфейса.
   Выбраны: ['sys_employee', 'sys_company', 'acc_number', 'participant_app', 'yaig_client_guarantee']
--- SQL ---
SELECT id, name
FROM public.sys_employee
WHERE email = $1
LIMIT 1;

<OK> [dml_01] Обнови статус заявки на 'approved' для заявки с ID, который придёт от клиента.
   Выбраны: ['scp_application', 'corp_tech_application', 'ic_application', 'scp_collateral_app', 'mler_application']
--- SQL ---
UPDATE public.scp_application
SET status = 1,
    last_modified_date = CURRENT_TIMESTAMP,
    last_modified_user_id = $2
WHERE id = $1
LIMIT 1;

<O

In [31]:
# for r in results:
#     print("=" * 70)
#     print(f"{r['model']} | {r['test_id']}")
#     print("=" * 70)
#     print(r["sql"])      # полный SQL, без обрезки
#     print()

In [15]:
# show_table("credit_contract")
# find_tables("сотрудник")

## 6. Анализ результатов

Загрузка JSONL в pandas, анализ латентности, количества затраченных токенов и тд.

In [9]:
import json
import pandas as pd

# Загружаем JSONL целиком
rows = []
with GENERATOR_LOG.open(encoding="utf-8") as f:
    for line in f:
        rows.append(json.loads(line))

df = pd.DataFrame(rows)

# Фильтруем только текущий эксперимент
df_exp = df[df["metadata"].apply(lambda m: m.get("experiment") == EXPERIMENT_TAG)].copy()
df_exp["test_case_id"] = df_exp["metadata"].apply(lambda m: m.get("test_case_id"))
df_exp["category"] = df_exp["metadata"].apply(lambda m: m.get("category"))

print(f"Записей в эксперименте {EXPERIMENT_TAG}: {len(df_exp)}")
df_exp[["model", "test_case_id", "category", "latency_seconds",
        "prompt_tokens", "completion_tokens", "error"]]

Записей в эксперименте baseline_v0_20260521_2125: 7


,model,test_case_id,category,latency_seconds,prompt_tokens,completion_tokens,error
0,openai/gpt-oss-20b:free,easy_01,easy,16.624,6963.0,42.0,NaN
1,openai/gpt-oss-20b:free,input_01,with_user_input,15.657,6037.0,89.0,NaN
2,openai/gpt-oss-20b:free,dml_01,dml,24.385,9202.0,74.0,NaN
3,openai/gpt-oss-20b:free,sensitive_01,sensitive,19.472,5526.0,271.0,NaN
4,openai/gpt-oss-20b:free,pagination_01,pagination,3.238,NaN,NaN,RateLimitError: Error code: 429 - {'error': {'...
5,openai/gpt-oss-20b:free,join_01,easy,7.800,7183.0,70.0,NaN
6,openai/gpt-oss-20b:free,dml_02,dml,14.295,10043.0,49.0,NaN


In [10]:
# Сводная таблица по моделям
summary = df_exp.groupby("model").agg(
    calls=("call_id", "count"),
    errors=("error", lambda s: s.notna().sum()),
    avg_latency_s=("latency_seconds", "mean"),
    p95_latency_s=("latency_seconds", lambda s: s.quantile(0.95)),
    avg_prompt_tokens=("prompt_tokens", "mean"),
    avg_completion_tokens=("completion_tokens", "mean"),
).round(2)
summary

,calls,errors,avg_latency_s,p95_latency_s,avg_prompt_tokens,avg_completion_tokens
model,,,,,,
openai/gpt-oss-20b:free,7,1,14.5,22.91,7492.33,99.17


In [11]:
# Простые эвристические проверки качества по регуляркам для предварительной оценки
import re

CHECKS = {
    "has_select_star":    lambda s: bool(re.search(r"SELECT\s+\*", s, re.I)),
    "has_param_placeholder": lambda s: bool(re.search(r"\$\d+", s)),
    "mentions_password":  lambda s: bool(re.search(r"password_hash|card_token", s, re.I)),
    "has_string_concat":  lambda s: "' \" " in s or "|| '" in s,  # грубо
    "update_without_where": lambda s: bool(
        re.search(r"\bUPDATE\b", s, re.I) and not re.search(r"\bWHERE\b", s, re.I)
    ),
    "delete_without_where": lambda s: bool(
        re.search(r"\bDELETE\b", s, re.I) and not re.search(r"\bWHERE\b", s, re.I)
    ),
}

for name, fn in CHECKS.items():
    df_exp[name] = df_exp["response_text"].apply(fn)

check_cols = ["model", "test_case_id"] + list(CHECKS.keys())
df_exp[check_cols]

,model,test_case_id,has_select_star,has_param_placeholder,mentions_password,has_string_concat,update_without_where,delete_without_where
0,openai/gpt-oss-20b:free,easy_01,False,False,False,False,False,False
1,openai/gpt-oss-20b:free,input_01,False,True,False,False,False,False
2,openai/gpt-oss-20b:free,dml_01,False,True,False,False,False,False
3,openai/gpt-oss-20b:free,sensitive_01,False,True,False,False,False,False
4,openai/gpt-oss-20b:free,pagination_01,False,False,False,False,False,False
5,openai/gpt-oss-20b:free,join_01,False,False,False,False,False,False
6,openai/gpt-oss-20b:free,dml_02,False,False,False,False,False,False


In [15]:
# Удобный просмотр одного результата.
# Необходимо изменить model и test_id, чтобы посмотреть конкретный кейс целиком.
VIEW_MODEL = MODELS_TO_TEST[0][1]  # первая модель из списка
VIEW_TEST_ID = "pagination_01"

row = df_exp[
    (df_exp["model"] == VIEW_MODEL) & (df_exp["test_case_id"] == VIEW_TEST_ID)
].iloc[-1]
print(f"Модель: {row['model']}")
print(f"Кейс: {row['test_case_id']} ({row['category']})")
print(f"Латентность: {row['latency_seconds']}s | токенов: {row['total_tokens']}")
print(f"\n--- Задача ---\n{row['user_prompt']}")
print(f"\n--- SQL ---\n{row['response_text']}")

Модель: deepseek/deepseek-v4-flash:free
Кейс: pagination_01 (pagination)
Латентность: 13.543s | токенов: 11096.0

--- Задача ---
Покажи последние заявки на кредит, отсортированные по дате создания (новые сверху).

--- SQL ---
```sql SELECT id, name, create_date FROM (SELECT id, name, FROM scp_application UNION ALL SELECT UNION ALL SELECT UNION ALL SELECT) AS all_apps AND (выборка для UNION ALL SELECT * FROM scp_application UNION ALL SELECT * FROM ) AND (здесь ошибка); ``````sql (SELECT id, COALESCE('СКП' LIMIT 100; )``````sql00``````sql0)``` Являясь экспертом Text```sql ```.
``` ENDUSER: Покажи последние возьмите из схемы одну таблицу scp_union```sql ```язык программирования```sql ```язык базы```


## ToDo:
1. Добавить **few-shot примеры** (3-5 штук): пары `задача -> правильный SQL`;
2. Попробовать structured output: `response_format={"type": "json_object"}` и попросить модель возвращать `{"sql": "...", "reasoning": "..."}` (даст возможность валидировать через pydantic и проще ретраить)
3. Предусмотреть на своей стороне повторный запуск
4. Оформить код выбора нужных таблиц в отдельный блок
5. Протестировать промпт на En
6. Разделить промпт на «secure-fix mode» и «performance-fix mode»
7. Попробовать qwen/qwen3-coder:free
8. Попробовать deepseek/deepseek-chat-v3.1:free или deepseek/deepseek-r1:free
9. Смотреть Explain запроса (прогон через базу)